# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

# Reading from Bronze layer

In [0]:
df = spark.table("data_lakehouse_project.bronze.crm_cust_info")

# Data Transformation

- Remove NULL 
- TRIM the string
- Normalization Marital Status and Gender
- Rename Columns

In [0]:
df.display()

## cst_id column remove NULLs + duplicate

In [0]:
df = df.dropDuplicates(["cst_id"])

In [0]:
df = df.filter(col("cst_id").isNotNull())

## Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(col("cst_marital_status") == "M", "Married")
         .when(col("cst_marital_status") == "S", "Single")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(col("cst_gndr") == "M", "Male")
         .when(col("cst_gndr") == "F", "Female")
         .otherwise("n/a")
    )
)

## Upper cst_key column + Standardized to match other table

In [0]:
df = df.withColumn("cst_key", F.regexp_replace(F.col("cst_key"), "-|NAS", ""))

## Rename Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("data_lakehouse_project.silver.crm_customers")
)

# Check queries

In [0]:
%sql
SELECT
    *
FROM data_lakehouse_project.silver.crm_customers